# Decision Trees & Random Forests

## Learning Objectives
- Understand how decision trees split data
- Learn about Gini impurity and entropy
- Implement tree visualization
- Master Random Forest ensemble method
- Understand feature importance
- Handle overfitting with pruning and regularization

## Prerequisites
- Classification and regression basics
- Understanding of overfitting

---
## Part 1: Decision Tree Fundamentals

### How Trees Make Decisions
A decision tree recursively partitions data based on feature thresholds:

1. Start with all data at root
2. Find the best feature and threshold to split
3. Create child nodes
4. Repeat until stopping criteria

### Splitting Criteria

**Gini Impurity** (classification):
$$G = 1 - \sum_{k=1}^{K} p_k^2$$

**Entropy** (classification):
$$H = -\sum_{k=1}^{K} p_k \log_2(p_k)$$

**MSE** (regression):
$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \bar{y})^2$$

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_wine, make_classification, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded!")

In [ ]:
# Visualize Gini vs Entropy
def gini(p):
    return 1 - p**2 - (1-p)**2

def entropy(p):
    if p == 0 or p == 1:
        return 0
    return -p * np.log2(p) - (1-p) * np.log2(1-p)

p_values = np.linspace(0.01, 0.99, 100)
gini_values = [gini(p) for p in p_values]
entropy_values = [entropy(p) for p in p_values]

plt.figure(figsize=(10, 6))
plt.plot(p_values, gini_values, 'b-', lw=2, label='Gini Impurity')
plt.plot(p_values, entropy_values, 'r-', lw=2, label='Entropy')
plt.xlabel('Probability of Class 1', fontsize=12)
plt.ylabel('Impurity', fontsize=12)
plt.title('Gini Impurity vs Entropy', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
plt.show()

print("Both measures:")
print("  • Maximum at p=0.5 (maximum uncertainty)")
print("  • Minimum at p=0 or p=1 (pure node)")

---
## Part 2: Building Your First Decision Tree

In [ ]:
# Load Iris dataset
iris = load_iris()
X_iris = iris.data[:, [2, 3]]  # Petal length and width only
y_iris = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

# Train decision tree
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_clf.fit(X_train, y_train)

print(f"Training Accuracy: {tree_clf.score(X_train, y_train):.2%}")
print(f"Test Accuracy: {tree_clf.score(X_test, y_test):.2%}")

In [ ]:
# Visualize the tree
plt.figure(figsize=(20, 10))
plot_tree(tree_clf, 
          feature_names=['petal_length', 'petal_width'],
          class_names=iris.target_names,
          filled=True, 
          rounded=True,
          fontsize=12)
plt.title('Decision Tree for Iris Classification', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize decision boundaries
def plot_decision_boundary(model, X, y, ax, title):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', edgecolors='black', s=50)
    ax.set_xlabel('Petal Length')
    ax.set_ylabel('Petal Width')
    ax.set_title(title)
    return scatter

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, depth in enumerate([1, 3, None]):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    acc = tree.score(X_test, y_test)
    title = f'max_depth={depth}\nTest Acc: {acc:.2%}'
    plot_decision_boundary(tree, X_train, y_train, axes[idx], title)

plt.suptitle('Effect of Tree Depth on Decision Boundary', fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 3: Overfitting and Regularization

Decision trees easily overfit! Control with:
- `max_depth`: Maximum tree depth
- `min_samples_split`: Minimum samples to split a node
- `min_samples_leaf`: Minimum samples in a leaf
- `max_features`: Number of features to consider for splitting

In [ ]:
# Demonstrate overfitting
X_complex, y_complex = make_classification(
    n_samples=200, n_features=10, n_informative=5,
    n_redundant=2, random_state=42
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_complex, y_complex, test_size=0.3, random_state=42
)

depths = range(1, 20)
train_scores = []
test_scores = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train_c, y_train_c)
    train_scores.append(tree.score(X_train_c, y_train_c))
    test_scores.append(tree.score(X_test_c, y_test_c))

plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores, 'b-o', label='Training')
plt.plot(depths, test_scores, 'r-s', label='Test')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree: Training vs Test Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axvspan(list(depths)[np.argmax(test_scores)-1], 20, alpha=0.1, color='red')
plt.annotate('Overfitting Zone', xy=(12, 0.85), fontsize=12, color='red')
plt.show()

print(f"Best test accuracy: {max(test_scores):.2%} at depth={depths[np.argmax(test_scores)]+1}")

In [ ]:
# Hyperparameter tuning with GridSearchCV
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_features': ['sqrt', 'log2', None]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_train_c, y_train_c)

print("Best Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest CV Score: {grid_search.best_score_:.4f}")
print(f"Test Score: {grid_search.score(X_test_c, y_test_c):.4f}")

---
## Part 4: Random Forest

### Ensemble Learning
Random Forest combines many decision trees:

1. **Bagging**: Each tree trained on bootstrap sample
2. **Random Features**: Each split considers random subset of features
3. **Aggregation**: Final prediction by voting (classification) or averaging (regression)

### Why It Works
- Reduces variance (overfitting)
- Different trees make different errors
- Averaging reduces noise

In [ ]:
# Compare single tree vs Random Forest
from sklearn.ensemble import RandomForestClassifier

# Single tree
tree_single = DecisionTreeClassifier(random_state=42)
tree_single.fit(X_train_c, y_train_c)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_c, y_train_c)

print("Single Tree vs Random Forest:")
print(f"  Single Tree - Train: {tree_single.score(X_train_c, y_train_c):.2%}, Test: {tree_single.score(X_test_c, y_test_c):.2%}")
print(f"  Random Forest - Train: {rf.score(X_train_c, y_train_c):.2%}, Test: {rf.score(X_test_c, y_test_c):.2%}")

In [ ]:
# Effect of number of trees
n_estimators_range = [1, 5, 10, 25, 50, 100, 200, 500]
rf_scores = []

for n in n_estimators_range:
    rf_temp = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf_temp.fit(X_train_c, y_train_c)
    rf_scores.append(rf_temp.score(X_test_c, y_test_c))

plt.figure(figsize=(10, 6))
plt.plot(n_estimators_range, rf_scores, 'g-o', lw=2, markersize=8)
plt.xlabel('Number of Trees')
plt.ylabel('Test Accuracy')
plt.title('Random Forest: Effect of Number of Trees')
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.show()

print(f"Best accuracy: {max(rf_scores):.2%} with {n_estimators_range[np.argmax(rf_scores)]} trees")

---
## Part 5: Feature Importance

Random Forest provides feature importance based on:
- Mean decrease in impurity across all trees
- How much each feature contributes to predictions

In [ ]:
# Load wine dataset for more features
wine = load_wine()
X_wine = wine.data
y_wine = wine.target

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

# Train Random Forest
rf_wine = RandomForestClassifier(n_estimators=100, random_state=42)
rf_wine.fit(X_train_w, y_train_w)

# Feature importance
importance_df = pd.DataFrame({
    'Feature': wine.feature_names,
    'Importance': rf_wine.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='forestgreen', alpha=0.8)
plt.xlabel('Feature Importance')
plt.title('Random Forest Feature Importance (Wine Dataset)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 5 Most Important Features:")
display(importance_df.head())

In [ ]:
# Feature selection using importance
threshold = 0.05
selected_features = importance_df[importance_df['Importance'] >= threshold]['Feature'].tolist()

print(f"Features with importance >= {threshold}:")
print(f"  {len(selected_features)} / {len(wine.feature_names)} features selected")

# Retrain with selected features only
feature_mask = [f in selected_features for f in wine.feature_names]
X_train_selected = X_train_w[:, feature_mask]
X_test_selected = X_test_w[:, feature_mask]

rf_selected = RandomForestClassifier(n_estimators=100, random_state=42)
rf_selected.fit(X_train_selected, y_train_w)

print(f"\nAll features - Test Accuracy: {rf_wine.score(X_test_w, y_test_w):.2%}")
print(f"Selected features - Test Accuracy: {rf_selected.score(X_test_selected, y_test_w):.2%}")

---
## Part 6: Decision Trees & Random Forest for Regression

In [ ]:
# Load diabetes dataset
diabetes = load_diabetes()
X_diab = diabetes.data
y_diab = diabetes.target

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diab, y_diab, test_size=0.2, random_state=42
)

# Train models
from sklearn.linear_model import Ridge

models = {
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Ridge Regression': Ridge(alpha=1.0)
}

print("Regression Model Comparison:")
print("=" * 50)
for name, model in models.items():
    model.fit(X_train_d, y_train_d)
    train_r2 = model.score(X_train_d, y_train_d)
    test_r2 = model.score(X_test_d, y_test_d)
    y_pred = model.predict(X_test_d)
    rmse = np.sqrt(mean_squared_error(y_test_d, y_pred))
    print(f"{name}:")
    print(f"  Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}, RMSE: {rmse:.2f}")

---
## Part 7: Boosting Algorithms

### Gradient Boosting
- Trees built sequentially
- Each tree corrects errors of previous trees
- More prone to overfitting than Random Forest

### AdaBoost
- Adjusts sample weights based on errors
- Misclassified samples get higher weights

In [ ]:
# Compare ensemble methods
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

ensemble_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []
for name, model in ensemble_models.items():
    model.fit(X_train, y_train)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    results.append({
        'Model': name,
        'Train Acc': model.score(X_train, y_train),
        'Test Acc': model.score(X_test, y_test),
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std()
    })

results_df = pd.DataFrame(results)
print("Ensemble Methods Comparison:")
display(results_df.round(4))

---
## Summary

| Method | Pros | Cons |
|--------|------|------|
| **Decision Tree** | Interpretable, fast | Overfits easily |
| **Random Forest** | Robust, feature importance | Less interpretable, slower |
| **Gradient Boosting** | Often best accuracy | Can overfit, slow to train |

### Key Hyperparameters

| Parameter | Effect |
|-----------|--------|
| `max_depth` | Controls tree depth (lower = less overfit) |
| `n_estimators` | Number of trees (more = better, diminishing returns) |
| `min_samples_split` | Minimum samples to split (higher = less overfit) |
| `max_features` | Features per split (lower = more diversity) |

In [ ]:
print("=" * 60)
print("Decision Trees & Random Forest Notebook Complete!")
print("=" * 60)
print("\nKey takeaways:")
print("  ✅ Decision trees split on features to minimize impurity")
print("  ✅ Single trees easily overfit - use regularization")
print("  ✅ Random Forests reduce variance through bagging")
print("  ✅ Feature importance helps with feature selection")
print("  ✅ Boosting can achieve higher accuracy but may overfit")